In [ ]:
!pip install pymupdf pytesseract pillow pandas

import fitz  # PyMuPDF
import pytesseract
from PIL import Image
import pandas as pd
import re
import uuid
import os

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 77.4 MB/s eta 0:00:00


In [ ]:
pdf_path = "/content/Receipt_May.pdf"   # <-- replace with your actual file

# 📌 Step 3: Configure Tesseract path (update if needed for Windows)
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# 📌 Step 4: Initialize storage
data = []

In [ ]:
doc = fitz.open(pdf_path)

for page_num in range(len(doc)):
    page = doc[page_num]

    # Try text extraction (works if PDF has real text)
    text = page.get_text("text")

    # If no text found → use OCR on image
    if not text.strip():
        pix = page.get_pixmap()
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        text = pytesseract.image_to_string(img)

    print(f"\n--- Page {page_num+1} Extracted Text ---\n{text[:500]}")  # Debug: show first 500 chars

    lines = text.split("\n")

    for line in lines:
        line = line.strip()
        if not line:
            continue

        # 📌 Regex for common transaction formats (adjustable)
        # Example: 01/09/2025 AMAZON 1200.50 Debit
        match = re.match(r"(\d{2}[/-]\d{2}[/-]\d{4})\s+(.+?)\s+(\d+[.,]?\d*)\s+(Credit|Debit|CR|DR)", line, re.IGNORECASE)

        if match:
            date, merchant, amount, txn_type = match.groups()

            # Normalize amount (replace , with . if needed)
            amount = amount.replace(",", "")

            # Normalize type
            if txn_type.upper() in ["CR", "CREDIT"]:
                txn_type = "Credit"
            else:
                txn_type = "Debit"

            # Unique ID for transaction
            transaction_id = str(uuid.uuid4())[:8]
            user_id = "U001"  # placeholder

            data.append([transaction_id, user_id, date, float(amount), merchant, txn_type])



--- Page 1 Extracted Text ---
Transaction Statement for 9359475651
01 Jan, 2025 - 28 Mar, 2025
Date
Transaction Details
Type
Amount
Mar 26, 2025
03:30 PM
CREDIT
₹30
Received from Rucha More
Transaction ID T2503261530309128480097
UTR No. 440284235471
Credited to
XXXXXX1604
Mar 24, 2025
10:09 AM
CREDIT
₹12
Received from Siddhi Didi
Transaction ID T2503241009132330029575
UTR No. 332121575027
Credited to
XXXXXX1604
Mar 18, 2025
05:59 PM
DEBIT
₹20
Paid to Siddhi Didi
Transaction ID T2503181759179112284250
UTR No. 247353938596
Pai

--- Page 2 Extracted Text ---
Date
Transaction Details
Type
Amount
Mar 16, 2025
10:23 AM
DEBIT
₹20
Paid to Siddhi Didi
Transaction ID T2503161023439636201168
UTR No. 692727642210
Paid by
XXXXXX1604
Mar 15, 2025
10:46 PM
DEBIT
₹200
Paid to OCEANIA  KSHETRIMAYUM
Transaction ID T2503152246122842161448
UTR No. 619198788989
Paid by
XXXXXX1604
Mar 15, 2025
02:27 PM
DEBIT
₹150
Paid to Nandini
Transaction ID T2503151427133399504980
UTR No. 464030756226
Paid by
XXXXXX1604

In [ ]:
df = pd.DataFrame(data, columns=["Transaction ID", "User ID", "Date", "Amount", "Merchant Name", "Type"])
df.to_csv("transactions.csv", index=False)

print("\n✅ Extraction complete. Transactions found:", len(df))
df.head()


✅ Extraction complete. Transactions found: 0


,Transaction ID,User ID,Date,Amount,Merchant Name,Type


In [ ]:
# Show all rows in notebook (careful if too many)
pd.set_option('display.max_rows', None)
df


,Transaction ID,User ID,Date,Amount,Merchant Name,Type


In [ ]:
import os
print("File saved at:", os.path.abspath("raw_transactions.csv"))


File saved at: /content/raw_transactions.csv


In [ ]:
import fitz  # PyMuPDF
import pytesseract
from PIL import Image
import pandas as pd
import os

pdf_path = "/content/Receipt_May.pdf"   # replace with your file
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

all_lines = []

doc = fitz.open(pdf_path)
for page_num in range(len(doc)):
    page = doc[page_num]

    # First try text mode
    text = page.get_text("text")

    # If empty, use OCR
    if not text.strip():
        pix = page.get_pixmap()
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        text = pytesseract.image_to_string(img)

    # Store every line
    lines = text.split("\n")
    for line in lines:
        line = line.strip()
        if line:
            all_lines.append([page_num+1, line])

# Save ALL raw text
df_raw = pd.DataFrame(all_lines, columns=["Page", "RawText"])
df_raw.to_csv("raw_transactions.csv", index=False)

print("✅ Saved raw extracted lines here:", os.path.abspath("raw_transactions.csv"))
print("Total lines extracted:", len(df_raw))


✅ Saved raw extracted lines here: /content/raw_transactions.csv
Total lines extracted: 1285


In [ ]:
import re
import pandas as pd
import fitz
import pytesseract
from PIL import Image

pdf_path = "/content/Receipt_May.pdf"   # your PDF file
pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

# Step 1: Extract raw text lines
all_lines = []
doc = fitz.open(pdf_path)
for page_num in range(len(doc)):
    page = doc[page_num]
    text = page.get_text("text")
    if not text.strip():  # fallback to OCR
        pix = page.get_pixmap()
        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        text = pytesseract.image_to_string(img)
    for line in text.split("\n"):
        line = line.strip()
        if line:
            all_lines.append(line)

# Step 2: Parse transactions
transactions = []
i = 0
while i < len(all_lines):
    line = all_lines[i]

    # Detect date (format: Mar 26, 2025)
    date_match = re.match(r"([A-Za-z]{3}\s+\d{1,2},\s+\d{4})", line)
    if date_match:
        date = date_match.group(1)
        i += 1 # Move to the next line

        # Check for time on the next line
        time = ""
        if i < len(all_lines) and ("AM" in all_lines[i] or "PM" in all_lines[i]):
            time = all_lines[i]
            i += 1 # Move to the next line

        # Check for transaction type on the next line
        txn_type = ""
        if i < len(all_lines) and all_lines[i].upper() in ["CREDIT", "DEBIT"]:
            txn_type = all_lines[i]
            i += 1 # Move to the next line

        # Check for amount line (contains '₹')
        amount = ""
        if i < len(all_lines) and "₹" in all_lines[i]:
            amount_line = all_lines[i]
            amount_match = re.search(r"₹\s*(\d[.,]?\d*)", amount_line)
            if amount_match:
                amount = amount_match.group(1).replace(",", "") # Extract and clean amount
            i += 1 # Move to the next line

        # Try to find merchant name (usually after amount or type, before Transaction ID)
        merchant = "Unknown Merchant" # Default
        while i < len(all_lines) and not "Transaction ID" in all_lines[i]:
             # Look for lines indicating merchant (e.g., "Received from", "Paid to")
            if "Received from" in all_lines[i] or "Paid to" in all_lines[i]:
                merchant = all_lines[i]
                break # Found merchant line
            i += 1 # Move to the next line

        # If we found a potential transaction, add it
        if date and txn_type and amount:
             # Unique ID for transaction
            transaction_id = str(uuid.uuid4())[:8]
            user_id = "U001"  # placeholder

            transactions.append([transaction_id, user_id, date, float(amount), merchant, txn_type])
        # If no transaction found after date, just continue to avoid infinite loop
        else:
            pass # Continue the loop to the next line


    else:
        i += 1 # Move to the next line if no date is matched

# Step 3: Create DataFrame and save
df = pd.DataFrame(transactions, columns=["Transaction_ID", "User_ID","Merchant/Source", "Amount", "Date/Timestamp",  "Type"])
df.to_csv("transactions.csv", index=False)

print("\n✅ Extraction complete. Transactions found:", len(df))
display(df.head())


✅ Extraction complete. Transactions found: 130


,Transaction ID,User ID,Date,Amount,Merchant Name,Type
0,a7f7a105,U001,"Mar 26, 2025",30.0,Received from Rucha More,CREDIT
1,9769ddcf,U001,"Mar 24, 2025",12.0,Received from Siddhi Didi,CREDIT
2,6630d9ee,U001,"Mar 18, 2025",20.0,Paid to Siddhi Didi,DEBIT
3,52dbdacb,U001,"Mar 17, 2025",1600.0,Paid to K K WAGH EDUCATION SOCIETY,DEBIT
4,432612d5,U001,"Mar 17, 2025",1655.0,Paid to,DEBIT


In [ ]:
# Save extracted transactions to CSV
output_file = "transactions.csv"
df.to_csv(output_file, index=False)

print(f"✅ All transactions saved to: {output_file}")


✅ All transactions saved to: transactions.csv


In [ ]:
# import pandas as pd

# # Example DataFrame (already extracted earlier)
# # df = pd.DataFrame(...)

# # Define category keywords
# categories = {
#     "Food": [
#         "swiggy","zomato","domino","pizza hut","mcdonald","roll","resto","sweets","kfc",
#         "subway","faasos","eatsure","biryani","haldiram","burger king","food","freshmenu",
#         "fresh","kissanconnect","chat","chaupati","street food","khaugalli","restaurant",
#         "bhojan","hotel","cafe"
#     ],
#     "Travel": [
#         "travels","travel","taxi","cab","auto","rickshaw","shuttle","tours","bus","red bus",
#         "uber","ola","drive","driver","blabla","quick ride","mahamandal"
#     ],
#     "Shopping": [
#         "mall","mart","footwear","bazar","dmart","store","shop","market","collection",
#         "trends","fashion","lifestyle","deals","essentials","grocery","supermarket",
#         "boutique","outlet","amazon","flipkart","myntra","ajio","snapdeal","meesho","nykaa",
#         "reliance","smart","zara","h&m"
#     ],
#     "Bills": [
#         "electricity","power supply","energy","lightbill","mseb","power","water","bill","bills",
#         "municipal","postpaid","mobile","prepaid","recharge","internet","data","dth","cable",
#         "set-top-box","airtel","tata sky","tata play","dish","tv","gas","property","house",
#         "maintenance","society charges","fastag","insurance","loan","credit card bill",
#         "netflix","prime","hotstar","bsnl","rent","tax"
#     ],
#     "Healthcare": [
#         "hospital","critical","ambulance","clinic","blood bank","doctor","icu","pharmacy",
#         "chemist","druggist","medical","medico","nursing home","health","health center",
#         "diagnostic","pathology","dispensary","wellness","green pharma","pharma","therapy",
#         "meds","medlife","medplus","dr.","apollo","24/7","medibuddy"
#     ]
# }


In [ ]:
# def classify_transaction(merchant_name):
#     if pd.isna(merchant_name):
#         return "Other"

#     name = merchant_name.lower()
#     for category, keywords in categories.items():
#         for keyword in keywords:
#             if keyword in name:
#                 return category
#     return "Other"


In [ ]:
# # Add Category column
# df["Category"] = df["Merchant Name"].apply(classify_transaction)
# # Show all rows
# pd.set_option('display.max_rows', None)

# # Show all columns
# pd.set_option('display.max_columns', None)

# # Now printing will show the full DataFrame
# print(df)


In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

In [ ]:
# # Save classified transactions to CSV
# output_file = "transactions_classified.csv"
# df.to_csv(output_file, index=False, encoding='utf-8-sig')

# print(f"✅ All classified transactions saved successfully to: {output_file}")


In [ ]:
# # --- 1. Total Income ---
# income = df.loc[df['Type'].str.upper() == 'CREDIT', 'Amount'].sum()
# print(f"💰 Total Income (Credits): {income}")

# # --- 2. Spending per Category ---
# spending = df.loc[df['Type'].str.upper() == 'DEBIT'].groupby('Category')['Amount'].sum()
# print("\n🛒 Spending by Category:")
# print(spending)

# # --- 3. (Optional) Save summary report to CSV ---
# summary = pd.DataFrame({
#     "Total Income": [income],
#     **{cat: [amt] for cat, amt in spending.items()}
# })
# summary.to_csv("financial_summary.csv", index=False)

# print("\n✅ Financial summary saved to 'financial_summary.csv'")
